# CUTLASS #3487 addressing regression
Pinned-source baseline; select a T4 GPU runtime and run all. A failing baseline is expected; no fix is applied.

In [ ]:
import pathlib, subprocess
def run(args, expected=0):
    p=subprocess.run(args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != expected: raise RuntimeError(f'{args[0]} exit {p.returncode}, expected {expected}')
    return p.stdout
env=run(['nvidia-smi'])+run(['nvcc','--version'])
pathlib.Path('cutlass3487-environment.log').write_text(env)
repo=pathlib.Path('/content/cutlass3487')
if not repo.exists(): run(['git','clone','--depth','1','https://github.com/NVIDIA/cutlass.git',str(repo)])
revision='0b55a2f691d69981583568fd9eb69687b1f0de8a'
run(['git','-C',str(repo),'fetch','--depth','1','origin',revision])
run(['git','-C',str(repo),'checkout','--detach',revision])
assert run(['git','-C',str(repo),'rev-parse','HEAD']).strip()==revision

In [ ]:
pathlib.Path("probe.cu").write_text("#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <initializer_list>\n#include \"cutlass/half.h\"\n#include \"cutlass/transform/pitch_linear_thread_map.h\"\n#include \"cutlass/transform/threadblock/predicated_tile_access_iterator.h\"\n\n#define CUDA_CHECK(call) do { auto status = (call); if (status != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(status)); std::exit(2); } } while (0)\n\n// Observe byte displacement without dereferencing a potentially incorrect address.\ntemplate <typename Element, int Rank>\n__global__ void offsets(Element *data, long long *actual, int extent_c, int extent_s,\n                        int move_c, int move_s, bool steady) {\n  using Shape = cutlass::layout::PitchLinearShape<32, 4>;\n  using Map = cutlass::transform::PitchLinearStripminedThreadMap<Shape, 32>;\n  using Iterator = cutlass::transform::threadblock::PredicatedTileAccessIterator<\n      Shape, Element, cutlass::layout::PitchLinear, Rank, Map,\n      cutlass::AlignedArray<Element, 1>>;\n  typename Iterator::Params params(cutlass::layout::PitchLinear(256));\n  // Keep backward residual transitions inside the allocated tensor.\n  Iterator it(params, data, {extent_c, extent_s}, threadIdx.x, {32, 4});\n  // First consume the residual tile to isolate the steady-state branch.\n  if (steady) it.add_tile_offset({Rank == 0 ? 1 : 0, Rank == 1 ? 1 : 0});\n  auto before = reinterpret_cast<std::intptr_t>(it.get());\n  it.add_tile_offset({move_c, move_s});\n  actual[threadIdx.x] = static_cast<long long>(reinterpret_cast<std::intptr_t>(it.get()) - before);\n}\n\ntemplate <typename Element, int Rank>\nint check_type(char const *name) {\n  Element *data;\n  long long *device_result, result[32];\n  CUDA_CHECK(cudaMalloc(&data, 256 * 256 * sizeof(Element)));\n  CUDA_CHECK(cudaMalloc(&device_result, sizeof(result)));\n  int failed = 0;\n  // Aligned and residual extents; zero, advance-only, cross-only and diagonal moves.\n  int const moves[][2] = {{0, 0}, {1, 0}, {0, 1}, {1, 1}};\n  for (bool steady : {false, true}) {\n    for (int extent : {64, 67}) {\n      for (auto const &move : moves) {\n        offsets<Element, Rank><<<1, 32>>>(data, device_result, extent, extent,\n                                         move[0], move[1], steady);\n        CUDA_CHECK(cudaGetLastError());\n        CUDA_CHECK(cudaDeviceSynchronize());\n        CUDA_CHECK(cudaMemcpy(result, device_result, sizeof(result), cudaMemcpyDeviceToHost));\n        // The first transition advances by the residual size, then whole tiles.\n        int dc = 32 * move[0], ds = 4 * move[1];\n        if (!steady) {\n          if (Rank == 0) dc += (extent % 32 ? extent % 32 : 32) - 32;\n          else ds += (extent % 4 ? extent % 4 : 4) - 4;\n        }\n        long long expected = (dc + 256LL * ds) * sizeof(Element);\n        bool pass = true;\n        for (auto value : result) pass = pass && value == expected;\n        failed += !pass;\n        std::printf(\"type=%s rank=%d steady=%d extent=%d move=(%d,%d) expected=%lld actual=%lld %s\\n\",\n                    name, Rank, steady, extent, move[0], move[1], expected, result[0], pass ? \"PASS\" : \"FAIL\");\n      }\n    }\n  }\n  CUDA_CHECK(cudaFree(device_result));\n  CUDA_CHECK(cudaFree(data));\n  return failed;\n}\n\nint main() {\n  int failed = check_type<cutlass::half_t, 0>(\"fp16\") + check_type<cutlass::half_t, 1>(\"fp16\")\n             + check_type<float, 0>(\"fp32\") + check_type<float, 1>(\"fp32\");\n  std::printf(\"failed=%d total=64\\n\", failed);\n  return failed ? 1 : 0;\n}\n")
run(['nvcc','-std=c++17','-O2','-arch=sm_75','-I'+str(repo/'include'),'probe.cu','-o','probe'])
p=subprocess.run(['./probe'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
pathlib.Path('cutlass3487-baseline.log').write_text(p.stdout)
print('baseline exit:',p.returncode)
assert p.returncode in (0,1), 'probe runtime failure'

In [ ]:
header=repo/"include/cutlass/transform/threadblock/predicated_tile_access_iterator.h"
s=header.read_text()
fixes=[["pointer_ += Shape::kStrided * tile_offset.strided() * sizeof_bits<Element>::value / 8;","add_pointer_offset(LongIndex(Shape::kStrided) * tile_offset.strided() * params_.stride_);"],["pointer_ += Shape::kContiguous * tile_offset.contiguous();","add_pointer_offset(LongIndex(Shape::kContiguous) * tile_offset.contiguous());"],["pointer_ += Shape::kStrided * tile_offset.strided();","add_pointer_offset(LongIndex(Shape::kStrided) * tile_offset.strided() * params_.stride_);"]]
for old,new in fixes:
    assert s.count(old)==1, old
    s=s.replace(old,new)
header.write_text(s)
run(["nvcc","-std=c++17","-O2","-arch=sm_75","-I"+str(repo/"include"),"probe.cu","-o","probe-fixed"])
result=run(["./probe-fixed"])
pathlib.Path("cutlass3487-fixed.log").write_text(result)
result=run(["compute-sanitizer","--tool","memcheck","--error-exitcode","99","./probe-fixed"])
pathlib.Path("cutlass3487-memcheck.log").write_text(result)

In [ ]:
test=repo/"test/unit/transform/threadblock/predicated_tile_access_iterator.cu"
test.write_text("/***************************************************************************************************\r\n * Copyright (c) 2017 - 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.\r\n * SPDX-License-Identifier: BSD-3-Clause\r\n *\r\n * Redistribution and use in source and binary forms, with or without\r\n * modification, are permitted provided that the following conditions are met:\r\n *\r\n * 1. Redistributions of source code must retain the above copyright notice, this\r\n * list of conditions and the following disclaimer.\r\n *\r\n * 2. Redistributions in binary form must reproduce the above copyright notice,\r\n * this list of conditions and the following disclaimer in the documentation\r\n * and/or other materials provided with the distribution.\r\n *\r\n * 3. Neither the name of the copyright holder nor the names of its\r\n * contributors may be used to endorse or promote products derived from\r\n * this software without specific prior written permission.\r\n *\r\n * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS \"AS IS\"\r\n * AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\r\n * IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\r\n * DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\r\n * FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\r\n * DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\r\n * SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\r\n * CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\r\n * OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\r\n * OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\r\n *\r\n **************************************************************************************************/\r\n\n#include \"../../common/cutlass_unit_test.h\"\n#include <cuda_runtime.h>\n#include <stdexcept>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <initializer_list>\n#include \"cutlass/half.h\"\n#include \"cutlass/transform/pitch_linear_thread_map.h\"\n#include \"cutlass/transform/threadblock/predicated_tile_access_iterator.h\"\n\n#define CUDA_CHECK(call) do { auto status = (call); if (status != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(status)); throw std::runtime_error(cudaGetErrorString(status)); } } while (0)\n\n// Observe byte displacement without dereferencing a potentially incorrect address.\ntemplate <typename Element, int Rank>\n__global__ void offsets(Element *data, long long *actual, int extent_c, int extent_s,\n                        int move_c, int move_s, bool steady) {\n  using Shape = cutlass::layout::PitchLinearShape<32, 4>;\n  using Map = cutlass::transform::PitchLinearStripminedThreadMap<Shape, 32>;\n  using Iterator = cutlass::transform::threadblock::PredicatedTileAccessIterator<\n      Shape, Element, cutlass::layout::PitchLinear, Rank, Map,\n      cutlass::AlignedArray<Element, 1>>;\n  typename Iterator::Params params(cutlass::layout::PitchLinear(256));\n  Iterator it(params, data, {extent_c, extent_s}, threadIdx.x);\n  // First consume the residual tile to isolate the steady-state branch.\n  if (steady) it.add_tile_offset({Rank == 0 ? 1 : 0, Rank == 1 ? 1 : 0});\n  auto before = reinterpret_cast<std::intptr_t>(it.get());\n  it.add_tile_offset({move_c, move_s});\n  actual[threadIdx.x] = static_cast<long long>(reinterpret_cast<std::intptr_t>(it.get()) - before);\n}\n\ntemplate <typename Element, int Rank>\nint check_type(char const *name) {\n  Element *data;\n  long long *device_result, result[32];\n  CUDA_CHECK(cudaMalloc(&data, 256 * 256 * sizeof(Element)));\n  CUDA_CHECK(cudaMalloc(&device_result, sizeof(result)));\n  int failed = 0;\n  // Aligned and residual extents; zero, advance-only, cross-only and diagonal moves.\n  int const moves[][2] = {{0, 0}, {1, 0}, {0, 1}, {1, 1}};\n  for (bool steady : {false, true}) {\n    for (int extent : {64, 67}) {\n      for (auto const &move : moves) {\n        offsets<Element, Rank><<<1, 32>>>(data, device_result, extent, extent,\n                                         move[0], move[1], steady);\n        CUDA_CHECK(cudaGetLastError());\n        CUDA_CHECK(cudaDeviceSynchronize());\n        CUDA_CHECK(cudaMemcpy(result, device_result, sizeof(result), cudaMemcpyDeviceToHost));\n        // The first transition advances by the residual size, then whole tiles.\n        int dc = 32 * move[0], ds = 4 * move[1];\n        if (!steady) {\n          if (Rank == 0) dc += (extent % 32 ? extent % 32 : 32) - 32;\n          else ds += (extent % 4 ? extent % 4 : 4) - 4;\n        }\n        long long expected = (dc + 256LL * ds) * sizeof(Element);\n        bool pass = true;\n        for (auto value : result) pass = pass && value == expected;\n        failed += !pass;\n        std::printf(\"type=%s rank=%d steady=%d extent=%d move=(%d,%d) expected=%lld actual=%lld %s\\n\",\n                    name, Rank, steady, extent, move[0], move[1], expected, result[0], pass ? \"PASS\" : \"FAIL\");\n      }\n    }\n  }\n  CUDA_CHECK(cudaFree(device_result));\n  CUDA_CHECK(cudaFree(data));\n  return failed;\n}\n\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp16Rank0) { EXPECT_EQ((check_type<cutlass::half_t, 0>(\"fp16\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp16Rank1) { EXPECT_EQ((check_type<cutlass::half_t, 1>(\"fp16\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp32Rank0) { EXPECT_EQ((check_type<float, 0>(\"fp32\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp32Rank1) { EXPECT_EQ((check_type<float, 1>(\"fp32\")), 0); }\n")
cm=repo/"test/unit/transform/threadblock/CMakeLists.txt"
s=cm.read_text()
s=s.replace("  predicated_tile_iterator.cu", "  predicated_tile_iterator.cu\n  predicated_tile_access_iterator.cu")
cm.write_text(s)
run(["cmake","-S",str(repo),"-B",str(repo/"build"),"-DCUTLASS_NVCC_ARCHS=75","-DCUTLASS_ENABLE_EXAMPLES=OFF","-DCUTLASS_ENABLE_TOOLS=ON","-DCUTLASS_ENABLE_LIBRARY=OFF"])
result=run(["cmake","--build",str(repo/"build"),"--target","cutlass_test_unit_transform_threadblock","-j2"])
pathlib.Path("cutlass3487-suite-build.log").write_text(result)
binary=next((repo/"build").rglob("cutlass_test_unit_transform_threadblock"))
result=run([str(binary)])
pathlib.Path("cutlass3487-suite.log").write_text(result)

In [ ]:
pathlib.Path("probe.cu").write_text("#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <initializer_list>\n#include \"cutlass/half.h\"\n#include \"cutlass/transform/pitch_linear_thread_map.h\"\n#include \"cutlass/transform/threadblock/predicated_tile_access_iterator.h\"\n\n#define CUDA_CHECK(call) do { auto status = (call); if (status != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(status)); std::exit(2); } } while (0)\n\n// Observe byte displacement without dereferencing a potentially incorrect address.\ntemplate <typename Element, int Rank>\n__global__ void offsets(Element *data, long long *actual, int extent_c, int extent_s,\n                        int move_c, int move_s, bool steady) {\n  using Shape = cutlass::layout::PitchLinearShape<32, 4>;\n  using Map = cutlass::transform::PitchLinearStripminedThreadMap<Shape, 32>;\n  using Iterator = cutlass::transform::threadblock::PredicatedTileAccessIterator<\n      Shape, Element, cutlass::layout::PitchLinear, Rank, Map,\n      cutlass::AlignedArray<Element, 1>>;\n  typename Iterator::Params params(cutlass::layout::PitchLinear(256));\n  // Keep backward residual transitions inside the allocated tensor.\n  Iterator it(params, data, {extent_c, extent_s}, threadIdx.x, {32, 4});\n  // First consume the residual tile to isolate the steady-state branch.\n  if (steady) it.add_tile_offset({Rank == 0 ? 1 : 0, Rank == 1 ? 1 : 0});\n  auto before = reinterpret_cast<std::intptr_t>(it.get());\n  it.add_tile_offset({move_c, move_s});\n  actual[threadIdx.x] = static_cast<long long>(reinterpret_cast<std::intptr_t>(it.get()) - before);\n}\n\ntemplate <typename Element, int Rank>\nint check_type(char const *name) {\n  Element *data;\n  long long *device_result, result[32];\n  CUDA_CHECK(cudaMalloc(&data, 256 * 256 * sizeof(Element)));\n  CUDA_CHECK(cudaMalloc(&device_result, sizeof(result)));\n  int failed = 0;\n  // Aligned and residual extents; zero, advance-only, cross-only and diagonal moves.\n  int const moves[][2] = {{0, 0}, {1, 0}, {0, 1}, {1, 1}};\n  for (bool steady : {false, true}) {\n    for (int extent : {64, 67}) {\n      for (auto const &move : moves) {\n        offsets<Element, Rank><<<1, 32>>>(data, device_result, extent, extent,\n                                         move[0], move[1], steady);\n        CUDA_CHECK(cudaGetLastError());\n        CUDA_CHECK(cudaDeviceSynchronize());\n        CUDA_CHECK(cudaMemcpy(result, device_result, sizeof(result), cudaMemcpyDeviceToHost));\n        // The first transition advances by the residual size, then whole tiles.\n        int dc = 32 * move[0], ds = 4 * move[1];\n        if (!steady) {\n          if (Rank == 0) dc += (extent % 32 ? extent % 32 : 32) - 32;\n          else ds += (extent % 4 ? extent % 4 : 4) - 4;\n        }\n        long long expected = (dc + 256LL * ds) * sizeof(Element);\n        bool pass = true;\n        for (auto value : result) pass = pass && value == expected;\n        failed += !pass;\n        std::printf(\"type=%s rank=%d steady=%d extent=%d move=(%d,%d) expected=%lld actual=%lld %s\\n\",\n                    name, Rank, steady, extent, move[0], move[1], expected, result[0], pass ? \"PASS\" : \"FAIL\");\n      }\n    }\n  }\n  CUDA_CHECK(cudaFree(device_result));\n  CUDA_CHECK(cudaFree(data));\n  return failed;\n}\n\nint main() {\n  int failed = check_type<cutlass::half_t, 0>(\"fp16\") + check_type<cutlass::half_t, 1>(\"fp16\")\n             + check_type<float, 0>(\"fp32\") + check_type<float, 1>(\"fp32\");\n  std::printf(\"failed=%d total=64\\n\", failed);\n  return failed ? 1 : 0;\n}\n")
test.write_text("/***************************************************************************************************\r\n * Copyright (c) 2017 - 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.\r\n * SPDX-License-Identifier: BSD-3-Clause\r\n *\r\n * Redistribution and use in source and binary forms, with or without\r\n * modification, are permitted provided that the following conditions are met:\r\n *\r\n * 1. Redistributions of source code must retain the above copyright notice, this\r\n * list of conditions and the following disclaimer.\r\n *\r\n * 2. Redistributions in binary form must reproduce the above copyright notice,\r\n * this list of conditions and the following disclaimer in the documentation\r\n * and/or other materials provided with the distribution.\r\n *\r\n * 3. Neither the name of the copyright holder nor the names of its\r\n * contributors may be used to endorse or promote products derived from\r\n * this software without specific prior written permission.\r\n *\r\n * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS \"AS IS\"\r\n * AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\r\n * IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\r\n * DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\r\n * FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\r\n * DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\r\n * SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\r\n * CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\r\n * OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\r\n * OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\r\n *\r\n **************************************************************************************************/\r\n\n#include \"../../common/cutlass_unit_test.h\"\n#include <cuda_runtime.h>\n#include <stdexcept>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <initializer_list>\n#include \"cutlass/half.h\"\n#include \"cutlass/transform/pitch_linear_thread_map.h\"\n#include \"cutlass/transform/threadblock/predicated_tile_access_iterator.h\"\n\n#define CUDA_CHECK(call) do { auto status = (call); if (status != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(status)); throw std::runtime_error(cudaGetErrorString(status)); } } while (0)\n\n// Observe byte displacement without dereferencing a potentially incorrect address.\ntemplate <typename Element, int Rank>\n__global__ void offsets(Element *data, long long *actual, int extent_c, int extent_s,\n                        int move_c, int move_s, bool steady) {\n  using Shape = cutlass::layout::PitchLinearShape<32, 4>;\n  using Map = cutlass::transform::PitchLinearStripminedThreadMap<Shape, 32>;\n  using Iterator = cutlass::transform::threadblock::PredicatedTileAccessIterator<\n      Shape, Element, cutlass::layout::PitchLinear, Rank, Map,\n      cutlass::AlignedArray<Element, 1>>;\n  typename Iterator::Params params(cutlass::layout::PitchLinear(256));\n  // Keep backward residual transitions inside the allocated tensor.\n  Iterator it(params, data, {extent_c, extent_s}, threadIdx.x, {32, 4});\n  // First consume the residual tile to isolate the steady-state branch.\n  if (steady) it.add_tile_offset({Rank == 0 ? 1 : 0, Rank == 1 ? 1 : 0});\n  auto before = reinterpret_cast<std::intptr_t>(it.get());\n  it.add_tile_offset({move_c, move_s});\n  actual[threadIdx.x] = static_cast<long long>(reinterpret_cast<std::intptr_t>(it.get()) - before);\n}\n\ntemplate <typename Element, int Rank>\nint check_type(char const *name) {\n  Element *data;\n  long long *device_result, result[32];\n  CUDA_CHECK(cudaMalloc(&data, 256 * 256 * sizeof(Element)));\n  CUDA_CHECK(cudaMalloc(&device_result, sizeof(result)));\n  int failed = 0;\n  // Aligned and residual extents; zero, advance-only, cross-only and diagonal moves.\n  int const moves[][2] = {{0, 0}, {1, 0}, {0, 1}, {1, 1}};\n  for (bool steady : {false, true}) {\n    for (int extent : {64, 67}) {\n      for (auto const &move : moves) {\n        offsets<Element, Rank><<<1, 32>>>(data, device_result, extent, extent,\n                                         move[0], move[1], steady);\n        CUDA_CHECK(cudaGetLastError());\n        CUDA_CHECK(cudaDeviceSynchronize());\n        CUDA_CHECK(cudaMemcpy(result, device_result, sizeof(result), cudaMemcpyDeviceToHost));\n        // The first transition advances by the residual size, then whole tiles.\n        int dc = 32 * move[0], ds = 4 * move[1];\n        if (!steady) {\n          if (Rank == 0) dc += (extent % 32 ? extent % 32 : 32) - 32;\n          else ds += (extent % 4 ? extent % 4 : 4) - 4;\n        }\n        long long expected = (dc + 256LL * ds) * sizeof(Element);\n        bool pass = true;\n        for (auto value : result) pass = pass && value == expected;\n        failed += !pass;\n        std::printf(\"type=%s rank=%d steady=%d extent=%d move=(%d,%d) expected=%lld actual=%lld %s\\n\",\n                    name, Rank, steady, extent, move[0], move[1], expected, result[0], pass ? \"PASS\" : \"FAIL\");\n      }\n    }\n  }\n  CUDA_CHECK(cudaFree(device_result));\n  CUDA_CHECK(cudaFree(data));\n  return failed;\n}\n\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp16Rank0) { EXPECT_EQ((check_type<cutlass::half_t, 0>(\"fp16\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp16Rank1) { EXPECT_EQ((check_type<cutlass::half_t, 1>(\"fp16\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp32Rank0) { EXPECT_EQ((check_type<float, 0>(\"fp32\")), 0); }\nTEST(Transform_Threadblock_PredicatedTileAccessIterator, TileOffsetFp32Rank1) { EXPECT_EQ((check_type<float, 1>(\"fp32\")), 0); }\n")
header.write_text(run(["git","-C",str(repo),"show",revision+":include/cutlass/transform/threadblock/predicated_tile_access_iterator.h"]))
run(["nvcc","-std=c++17","-O2","-arch=sm_75","-I"+str(repo/"include"),"probe.cu","-o","probe"])
result=run(["./probe"], expected=1)
pathlib.Path("cutlass3487-baseline-final.log").write_text(result)
run(["cmake","--build",str(repo/"build"),"--target","cutlass_test_unit_transform_threadblock","-j2"])
result=run([str(binary)], expected=1)
pathlib.Path("cutlass3487-suite-baseline.log").write_text(result)
s=header.read_text()
for old,new in fixes:
    assert s.count(old)==1
    s=s.replace(old,new)
header.write_text(s)
run(["nvcc","-std=c++17","-O2","-arch=sm_75","-I"+str(repo/"include"),"probe.cu","-o","probe-fixed"])
result=run(["./probe-fixed"])
pathlib.Path("cutlass3487-fixed-final.log").write_text(result)
result=run(["compute-sanitizer","--tool","memcheck","--error-exitcode","99","./probe-fixed"])
pathlib.Path("cutlass3487-memcheck-final.log").write_text(result)
run(["cmake","--build",str(repo/"build"),"--target","cutlass_test_unit_transform_threadblock","-j2"])
result=run([str(binary)])
pathlib.Path("cutlass3487-suite-fixed.log").write_text(result)